In [8]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Conv1D, Flatten, concatenate, Dropout, Multiply, Lambda
from tensorflow.keras.layers import BatchNormalization, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df= pd.read_csv("UNSW_augmented_filtered_data.csv")
df.shape

(257673, 44)

In [3]:
import pickle
import os
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, auc, accuracy_score)
from scipy.stats import ttest_rel
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, LSTM, Conv1D, Flatten, concatenate, Dropout,
                                     Multiply, Reshape, BatchNormalization, GlobalAveragePooling1D,
                                     Lambda)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
selected_features=['dur','proto','service','state','sbytes','dbytes','dload','sloss','sinpkt',
                   'sjit','swin','stcpb','dtcpb','dwin','synack','ackdat','dmean','trans_depth',
                   'ct_state_ttl','ct_dst_src_ltm','ct_ftp_cmd','ct_src_ltm']

In [5]:
df_selected=df[selected_features]
df_selected.shape

(257673, 22)

In [6]:
X = df_selected.values
y = df['attack_cat'].values

In [9]:
# Encode the labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_encoded = pd.get_dummies(y_encoded).values  # One-hot encode for multiclass classification

In [18]:
# Define CNN Model
def create_cnn_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)  # Output layer for multiclass classification
    model = Model(inputs, output)
    return model
# Define LSTM Model
def create_lstm_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = LSTM(64, return_sequences=True)(inputs)
    x = BatchNormalization()(x)
    x = LSTM(32)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)  # Output layer for multiclass classification
    model = Model(inputs, output)
    return model
# Define FNN Model
def create_fnn_model(input_shape, num_classes):
    inputs = Input(shape=(input_shape,))
    x = Dense(128, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)  # Output layer for multiclass classification
    model = Model(inputs, output)
    return model
# Define attention mechanism
def attention_mechanism(inputs):
    attention_weights = Dense(inputs.shape[-1], activation='softmax', name='attention_weights')(inputs)
    attention_output = Multiply(name='attention_output')([inputs, attention_weights])
    return attention_output, attention_weights



In [19]:
from memory_profiler import memory_usage

In [20]:
def train_ensemble():
    return ensemble_model.fit(
        [X_train_cnn, X_train_cnn, X_train], y_train,
        epochs=50,
        batch_size=128,
        validation_split=0.1,
        verbose=0,
        validation_data=([X_test_cnn, X_test_cnn, X_test], y_test)
    )

In [22]:

# Convert one-hot labels back to class labels for StratifiedKFold
y_labels = np.argmax(y_encoded, axis=1)
# --------------------------------- Part 4: 5-Fold Cross-Validation Setup ---------------------------------
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

ensemble_accs, cnn_accs, lstm_accs, fnn_accs = [], [], [], []
ensemble_times, cnn_times, lstm_times, fnn_times = [], [], [], []
ensemble_memory_usages, cnn_memory_usages, lstm_memory_usages, fnn_memory_usages = [], [], [], []
all_y_test = []
all_ensemble_pred = []
all_cnn_pred=[]
all_lstm_pred=[]
all_fnn_pred=[]
all_ensemble_prob = []
all_cm_ensemble = []
all_cm_cnn = []
all_cm_lstm = []
all_cm_fnn = []
attention_weights=[]
avg_atts=[]
fold = 1
for train_idx, test_idx in kfold.split(X, y_labels):
    print(f"\n=== Fold {fold} ===")
    fold += 1

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
    # Number of classes in the dataset
    num_classes = y_train.shape[1]
    # CNN
    cnn_model = create_cnn_model(X_train_cnn.shape[1:], num_classes)
    cnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    cnn_mem_usage, cnn_history = memory_usage(
    (cnn_model.fit, (X_train_cnn, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
    interval=0.1,
    retval=True )
    #cnn_history = cnn_model.fit(X_train_cnn, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    cnn_times.append(end - start)
    cnn_peak_memory = max(cnn_mem_usage)
    cnn_memory_usages.append(cnn_peak_memory)
    cnn_pred = (cnn_model.predict(X_test_cnn,verbose=0) > 0.5).astype(int)
    cnn_accs.append(accuracy_score(y_test, cnn_pred))
    
    # LSTM
  
    lstm_model = create_lstm_model(X_train_cnn.shape[1:], num_classes)
    lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    lstm_mem_usage, lstm_history = memory_usage(
    (lstm_model.fit, (X_train_cnn, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
     interval=0.1,retval=True)
    #lstm_history= lstm_model.fit(X_train_cnn, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    lstm_times.append(end - start)
    lstm_peak_memory = max(lstm_mem_usage)
    lstm_memory_usages.append(lstm_peak_memory)
    lstm_pred = (lstm_model.predict(X_test_cnn,verbose=0) > 0.5).astype(int)
    lstm_accs.append(accuracy_score(y_test, lstm_pred))
    
     # FNN
    
    fnn_model = create_fnn_model(X_train.shape[1], num_classes)
    fnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    fnn_mem_usage, fnn_history = memory_usage(
    (fnn_model.fit, (X_train, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
    interval=0.1,
    retval=True)
    #fnn_history=fnn_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    fnn_times.append(end - start)
    fnn_peak_memory = max(fnn_mem_usage)
    fnn_memory_usages.append(fnn_peak_memory)
    fnn_pred = (fnn_model.predict(X_test,verbose=0) > 0.5).astype(int)
    fnn_accs.append(accuracy_score(y_test, fnn_pred))
    
     # Ensemble
   
    # Get validation accuracy for dynamic weighting
    cnn_val_accuracy = cnn_model.evaluate(X_test_cnn, y_test, verbose=0)[1]
    lstm_val_accuracy = lstm_model.evaluate(X_test_cnn, y_test, verbose=0)[1]
    fnn_val_accuracy = fnn_model.evaluate(X_test, y_test, verbose=0)[1]
    
    # Calculate dynamic weights based on validation accuracy
    total_accuracy = cnn_val_accuracy + lstm_val_accuracy + fnn_val_accuracy
    weights = {
    'cnn': cnn_val_accuracy / total_accuracy,
    'lstm': lstm_val_accuracy / total_accuracy,
    'fnn': fnn_val_accuracy / total_accuracy
    }
    # Weighted outputs using Lambda layers
    cnn_weighted_output = Lambda(lambda x: x * weights['cnn'])(cnn_model.output)
    lstm_weighted_output = Lambda(lambda x: x * weights['lstm'])(lstm_model.output)
    fnn_weighted_output = Lambda(lambda x: x * weights['fnn'])(fnn_model.output)
    # Combine weighted outputs
    combined = concatenate([cnn_weighted_output, lstm_weighted_output, fnn_weighted_output], name='combined_features')
    # Apply attention mechanism on combined output
    attention_output, attention_tensor = attention_mechanism(combined)
    # Final output layer for multiclass classification
    output = Dense(num_classes, activation='softmax', name='output_layer')(attention_output)
    
    # Directly combine the model outputs without attention for testing
    combined = concatenate([cnn_weighted_output, lstm_weighted_output, fnn_weighted_output], name='combined_features')



   # Define the complete ensemble model
    ensemble_model = Model(inputs=[cnn_model.input, lstm_model.input, fnn_model.input], outputs=output)
    # This returns both the final classification output and attention weights
    ensemble_model_full = Model(inputs=[cnn_model.input, lstm_model.input, fnn_model.input],
                            outputs=[output, attention_tensor])

    # Compile and train the simplified model
    ensemble_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    ensemble_mem_usage, ensemble_history = memory_usage(
    train_ensemble,interval=0.1,retval=True)
    #ensemble_history=ensemble_model.fit(
     #   [X_train_cnn, X_train_cnn, X_train], y_train,
       # validation_data=([X_test_cnn, X_test_cnn, X_test], y_test),
        #epochs=50, batch_size=128, verbose=0)
    ensemble_peak_memory = max(ensemble_mem_usage)
    ensemble_memory_usages.append(ensemble_peak_memory)
    end = time.time()
    ensemble_times.append(end - start)
    ensemble_output, att_weights = ensemble_model_full.predict([X_test_cnn, X_test_cnn, X_test], verbose=0)
    ensemble_pred = (ensemble_output > 0.5).astype(int)
   #ensemble_pred = (ensemble_model.predict([X_test_cnn, X_test_cnn, X_test],verbose=0) > 0.5).astype(int)
    ensemble_accs.append(accuracy_score(y_test, ensemble_pred))
    all_y_test.append(y_test)
    all_ensemble_pred.append(ensemble_pred)
    all_ensemble_prob.append(ensemble_model.predict([X_test_cnn, X_test_cnn, X_test],verbose=0))  # Probabilities for ROC/PR
    ensemble_probs, att_weights = ensemble_model_full.predict([X_test_cnn, X_test_cnn, X_test], verbose=0)
    attention_weights.append(att_weights)  # Now att_weights is a real NumPy array
    avg_atts.append(np.mean(att_weights, axis=0))  # Mean over all test samples
    y_pred_ensemble = ensemble_model.predict([X_test_cnn, X_test_cnn, X_test],verbose=0)
    y_pred_classes_ensemble = np.argmax(y_pred_ensemble, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)
    cm_ensemble = confusion_matrix(y_true_classes, y_pred_classes_ensemble)
    all_cm_ensemble.append(cm_ensemble)
    y_pred_classes_cnn = np.argmax(cnn_pred, axis=1)
    cm_cnn=confusion_matrix(y_true_classes,  y_pred_classes_cnn)
    all_cm_cnn.append(cm_cnn)
    y_pred_classes_lstm = np.argmax(lstm_pred, axis=1)
    cm_lstm=confusion_matrix(y_true_classes,y_pred_classes_lstm)
    all_cm_lstm.append(cm_lstm)
    y_pred_classes_fnn = np.argmax(fnn_pred, axis=1)
    cm_fnn=confusion_matrix(y_true_classes,  y_pred_classes_fnn)
    all_cm_fnn.append(cm_fnn)
    


=== Fold 1 ===

=== Fold 2 ===

=== Fold 3 ===

=== Fold 4 ===

=== Fold 5 ===


In [46]:

# --------------------------------- Part 5: Report Results ---------------------------------
def report_scores(name, scores, times, memories):
    print(f"{name}: Accuracy = {np.mean(scores):.4f} ± {np.std(scores):.4f}, "
          f"Time = {np.mean(times):.2f}s ± {np.std(times):.2f}s, "
          f"Memory = {np.mean(memories):.2f} MiB ± {np.std(memories):.2f} MiB")

print("\n=== 5-Fold Cross-validation Results ===")
report_scores("CNN", cnn_accs, cnn_times, cnn_memory_usages)
report_scores("LSTM", lstm_accs, lstm_times, lstm_memory_usages)
report_scores("FNN", fnn_accs, fnn_times, fnn_memory_usages)
report_scores("Ensemble", ensemble_accs, ensemble_times, ensemble_memory_usages)


=== 5-Fold Cross-validation Results ===
CNN: Accuracy = 0.8254 ± 0.0104, Time = 612.52s ± 81.30s, Memory = 1634.22 MiB ± 113.89 MiB
LSTM: Accuracy = 0.8327 ± 0.0298, Time = 3528.70s ± 570.63s, Memory = 1639.87 MiB ± 109.44 MiB
FNN: Accuracy = 0.8549 ± 0.0093, Time = 360.97s ± 27.21s, Memory = 1629.44 MiB ± 105.87 MiB
Ensemble: Accuracy = 0.8937 ± 0.0258, Time = 3261.76s ± 282.17s, Memory = 1721.63 MiB ± 102.96 MiB


In [47]:

# --------------------------------- Part 6: Statistical Significance Testing ---------------------------------
print("\n=== Paired t-tests ===")
print("Ensemble vs CNN:", ttest_rel(ensemble_accs, cnn_accs))
print("Ensemble vs LSTM:", ttest_rel(ensemble_accs, lstm_accs))
print("Ensemble vs FNN:", ttest_rel(ensemble_accs, fnn_accs))


=== Paired t-tests ===
Ensemble vs CNN: TtestResult(statistic=4.691803927046335, pvalue=0.00936519033602254, df=4)
Ensemble vs LSTM: TtestResult(statistic=3.2511954720882987, pvalue=0.03134030421548158, df=4)
Ensemble vs FNN: TtestResult(statistic=4.474573880392635, pvalue=0.01103578194498423, df=4)


In [49]:
class_names = ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers',
              'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']

In [52]:
report = classification_report(y_true_classes , y_pred_classes_ensemble, target_names=class_names)
print("Ablation-without WGANGP - Ensemble Model (5-Fold CV):UNSW-NB15-Multiclass Classification")
print(report)

Ablation-without WGANGP - Ensemble Model (5-Fold CV):UNSW-NB15-Multiclass Classification
                precision    recall  f1-score   support

      Analysis       0.26      0.82      0.39      2677
      Backdoor       0.40      0.88      0.56      2329
           DoS       0.87      0.89      0.88     16353
      Exploits       0.97      0.94      0.95     44525
       Fuzzers       0.96      0.93      0.95     24246
       Generic       0.95      0.85      0.89     58871
        Normal       0.97      0.88      0.93     93000
Reconnaissance       0.81      0.91      0.86     13987
     Shellcode       0.38      0.85      0.53      1511
         Worms       0.00      0.14      0.01       174

      accuracy                           0.89    257673
     macro avg       0.66      0.81      0.69    257673
  weighted avg       0.93      0.89      0.91    257673

